In [1]:
!pip -q install unsloth

# Specific versions that work together without conflicts
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install pymupdf

# datasets: for loading our JSONL files cleanly
!pip -q install -U datasets

print("✓ All libraries installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.

In [2]:
import os
import gc
import re
import time
import json
import warnings
from typing import List

warnings.filterwarnings("ignore")

import torch
from datasets import Dataset, load_dataset

# Unsloth's fast model loader — handles 4-bit QLoRA automatically
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
import unicodedata
import warnings
from typing import List, Dict, Any
import fitz  # PyMuPDF

# ============================================================
# CELL 2: Import everything we need
# ============================================================


# ── GPU check ──────────────────────────────────────────────
# If this fails → Runtime → Change runtime type → T4 GPU
assert torch.cuda.is_available(), "No GPU found! Go to Runtime → Change runtime type → T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"✓ GPU: {gpu_name}")
print(f"✓ VRAM available: {vram_gb:.1f} GB")
print(f"✓ BF16 supported: {is_bfloat16_supported()}")



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✓ GPU: Tesla T4
✓ VRAM available: 14.6 GB
✓ BF16 supported: False


In [14]:
# -------------------------
#  Real file paths
# -------------------------
instruction_data_path = "/content/nexora_sft_dataset.jsonl"
for path in [ instruction_data_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")

# -------------------------
#  Simple config
# -------------------------
BASE_MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

MAX_SEQ_LENGTH = 1024
SEED = 42

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8

# SFT
SFT_LR = 5e-5
SFT_EPOCHS = 5


WARMUP_RATIO = 0.05
LOGGING_STEPS = 1

OUTPUT_ROOT = "/content/NEXORA"


STAGE1_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage1_instruction_adapter"
STAGE1_MERGED_DIR  = f"{OUTPUT_ROOT}/stage1_instruction_merged_model"
STAGE1_gguf_DIR    = f"{OUTPUT_ROOT}/stage1_instruction_model_gguf"


for path in [
    OUTPUT_ROOT,
    STAGE1_ADAPTER_DIR,
    STAGE1_MERGED_DIR,
    STAGE1_gguf_DIR
]:
    os.makedirs(path, exist_ok=True)

In [6]:
# -------------------------
#  Helper functions
# -------------------------
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result

In [7]:
def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"

In [8]:
def generate_answer(
    model,
    tokenizer,
    instruction: str,
    input_text: str = "",
    max_new_tokens: int = 100,
):
    FastLanguageModel.for_inference(model)

    instruction = str(instruction).strip()
    input_text = str(input_text or "").strip()

    # Build the user message
    if input_text:
        user_content = f"{instruction}\n\n{input_text}"
    else:
        user_content = instruction

    messages = [
        {
            "role": "user",
            "content": user_content,
        }
    ]

    # Use Qwen's native chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,

            # Keep factual answers reasonably short
            max_new_tokens=max_new_tokens,

            # Deterministic decoding for factual QA
            do_sample=False,

            # Stop / padding
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode ONLY tokens generated after the prompt
    input_length = inputs["input_ids"].shape[-1]

    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return answer.strip()

In [9]:
def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    This is used at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    - Stage 2 loads STAGE1_MERGED_DIR
    - Stage 3 loads STAGE2_MERGED_DIR
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer


In [10]:
def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

In [11]:
import torch
from datasets import Dataset, load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported


# ============================================================
# STAGE 1 DATA: SFT Instruction JSONL
# ============================================================

print("\n==============================")
print("STAGE 1: INSTRUCTION DATA")
print("==============================")

# Load the base model and tokenizer here to make tokenizer available
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train",
    encoding="latin-1",
)

# Validate required columns
required_instruction_cols = {"instruction", "output"}

missing_cols = (
    required_instruction_cols
    - set(instruction_dataset.column_names)
)

if missing_cols:
    raise ValueError(
        f"Instruction dataset missing columns: {missing_cols}"
    )


# ------------------------------------------------------------
# Format dataset using Qwen chat template
# ------------------------------------------------------------

def format_instruction_record(example):

    instruction = str(
        example.get("instruction", "")
    ).strip()

    input_text = str(
        example.get("input", "") or ""
    ).strip()

    output = str(
        example.get("output", "")
    ).strip()

    # Build user content
    if input_text:
        user_content = f"{instruction}\n\n{input_text}"
    else:
        user_content = instruction

    messages = [
        {
            "role": "user",
            "content": user_content,
        },
        {
            "role": "assistant",
            "content": output,
        },
    ]

    # Qwen2.5 native chat format
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}


# ------------------------------------------------------------
# Create Stage 1 dataset
# ------------------------------------------------------------

stage1_dataset = instruction_dataset.map(
    format_instruction_record
)


# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("Instruction rows:", len(stage1_dataset))

print("\n==============================")
print("SAMPLE TRAINING TEXT")
print("==============================")

print(stage1_dataset[0]["text"])



STAGE 1: INSTRUCTION DATA
==((====))==  Unsloth 2026.8.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1298 [00:00<?, ? examples/s]

Instruction rows: 1298

SAMPLE TRAINING TEXT
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
How many hours is a standard full-time workweek at Nexora?<|im_end|>
<|im_start|>assistant
Full-time employees are engaged for a standard workweek of forty (40) hours.<|im_end|>



In [13]:

# ============================================================
# STAGE 1: Load Stage 1 base model -> instruction SFT
# ============================================================


stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

FastLanguageModel.for_training(stage1_model)
tokenizer.padding_side = "right"

stage1_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    # Training duration
    num_train_epochs=SFT_EPOCHS,  # 5 epochs

    # Batch
    per_device_train_batch_size=BATCH_SIZE,  # 4
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,  # 8

    # Optimization
    learning_rate=SFT_LR,  # 5e-5
    warmup_ratio=WARMUP_RATIO,  # 0.05
    optim="adamw_8bit",

    # Logging / saving
    logging_steps=LOGGING_STEPS,  # 1
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",

    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    # Dataset
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,  # 1024
    packing=False,

    # Reproducibility
    seed=SEED,
)


==((====))==  Unsloth 2026.8.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.8.3 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [15]:
stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=stage1_dataset,
    args=stage1_config,)
train_and_measure(stage1_trainer, "STAGE 1 - INSTRUCTION FINE-TUNING")


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1298 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,298 | Num Epochs = 5 | Total steps = 205
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,3.953200
2,3.799500
3,3.859400
4,3.714100
5,3.418100
6,3.403500
7,3.233400
8,3.226600
9,3.004200
10,2.930400



STAGE 1 - INSTRUCTION FINE-TUNING RESULTS
Train time/sec: 879.56
Peak allocated VRAM/GB: 2.741
Peak reserved VRAM/GB: 2.85


TrainOutput(global_step=205, training_loss=1.1964248497311663, metrics={'train_runtime': 873.6389, 'train_samples_per_second': 7.429, 'train_steps_per_second': 0.235, 'total_flos': 4542162400404480.0, 'train_loss': 1.1964248497311663, 'epoch': 5.0})

In [18]:
save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE1_ADAPTER_DIR,
    merged_dir=STAGE1_MERGED_DIR,
    stage_name="Stage 1",
)

del stage1_trainer
del stage1_model
clear_gpu_memory()


Saving Stage 1 adapter...
Stage 1 adapter saved to: /content/NEXORA/stage1_instruction_adapter

Merging Stage 1 adapter with base model...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:02<00:00, 62.39s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:02<00:00, 62.04s/it]


Unsloth: Merge process complete. Saved to `/content/NEXORA/stage1_instruction_merged_model`
Stage 1 merged model saved to: /content/NEXORA/stage1_instruction_merged_model


In [19]:
# ============================================================
# LOAD TRAINED STAGE 1 MODEL FOR EVALUATION
# ============================================================

stage1_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE1_MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(stage1_model)

tokenizer.padding_side = "right"

print("Loaded trained Stage 1 merged model:")
print(STAGE1_MERGED_DIR)

==((====))==  Unsloth 2026.8.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loaded trained Stage 1 merged model:
/content/NEXORA/stage1_instruction_merged_model


In [20]:
# ============================================================
# CONTEXT-GROUNDED SFT TEST
# ============================================================

CTX_INSTR = """Answer the employee's question using only the provided policy context.
If the answer is not contained in the context, state that the provided policy context does not specify it.
Do not use outside knowledge."""


test_cases = [
    # 1. Direct context extraction
    {
        "name": "Direct Fact",
        "context": """
Full-time employees are entitled to twelve (12) days of
Sick Leave per calendar year.
""",
        "question": "How many Sick Leave days do full-time employees get?",
        "expected": "12 days"
    },

    # 2. Context vs model's prior/wrong memory
    {
        "name": "Context Conflict",
        "context": """
Full-time employees are entitled to twelve (12) days of
Sick Leave per calendar year.
""",
        "question": "I heard employees only get 10 Sick Leave days. Is that correct?",
        "expected": "No; 12 days"
    },

    # 3. Probation direct
    {
        "name": "Probation Fact",
        "context": """
All new full-time hires are subject to a probationary period
of ninety (90) calendar days from their date of joining.
""",
        "question": "How long is the probationary period?",
        "expected": "90 calendar days"
    },

    # 4. Probation reasoning
    {
        "name": "Probation Reasoning",
        "context": """
All new full-time hires are subject to a probationary period
of ninety (90) calendar days from their date of joining.
""",
        "question": "I joined 45 days ago. Am I still probationary?",
        "expected": "Yes; 45 < 90"
    },

    # 5. Probation boundary
    {
        "name": "Probation Boundary",
        "context": """
All new full-time hires are subject to a probationary period
of ninety (90) calendar days from their date of joining.
""",
        "question": "I joined 95 days ago. Am I still probationary?",
        "expected": "No; probation is 90 days"
    },

    # 6. Maternity direct
    {
        "name": "Maternity Fact",
        "context": """
Female employees who have completed at least eighty (80)
days of continuous employment are entitled to twenty-six
(26) weeks of paid Maternity Leave for the first two childbirths.
""",
        "question": "What is the maternity leave entitlement for the first childbirth?",
        "expected": "26 weeks"
    },

    # 7. Eligibility reasoning
    {
        "name": "Eligibility",
        "context": """
Female employees who have completed at least eighty (80)
days of continuous employment are entitled to twenty-six
(26) weeks of paid Maternity Leave for the first two childbirths.
""",
        "question": "I have worked here for 85 days and I'm expecting my first child. Am I eligible?",
        "expected": "Yes; 85 >= 80, 26 weeks"
    },

    # 8. Eligibility failure
    {
        "name": "Eligibility Negative",
        "context": """
Female employees who have completed at least eighty (80)
days of continuous employment are entitled to twenty-six
(26) weeks of paid Maternity Leave for the first two childbirths.
""",
        "question": "I have worked here for 60 days. Am I eligible for this maternity leave?",
        "expected": "No; minimum is 80 days"
    },

    # 9. Numerical reasoning
    {
        "name": "Leave Calculation",
        "context": """
Employees are entitled to eighteen (18) days of Annual Leave
per calendar year, accrued at the rate of 1.5 days per month.
""",
        "question": "I joined 4 months ago. How much Annual Leave have I accrued?",
        "expected": "6 days"
    },

    # 10. Numerical reasoning
    {
        "name": "Insurance Calculation",
        "context": """
A Group Term Life Insurance plan provides coverage equal to
five times the employee's annual CTC.
""",
        "question": "My annual CTC is INR 10,00,000. What is my insurance coverage?",
        "expected": "INR 50,00,000"
    },

    # 11. Abstention
    {
        "name": "Missing Information",
        "context": """
Employees receive health insurance and Group Term Life
Insurance benefits.
""",
        "question": "Do employees receive free Netflix subscriptions?",
        "expected": "Context does not specify"
    },

    # 12. Hallucination resistance
    {
        "name": "Hallucination Test",
        "context": """
The standard full-time workweek is forty (40) hours.
""",
        "question": "Does Nexora provide employees with a free gym membership?",
        "expected": "Context does not specify"
    },

    # 13. Question contains false assumption
    {
        "name": "False Assumption",
        "context": """
The standard full-time workweek is forty (40) hours.
""",
        "question": "Since Nexora has a 35-hour workweek, how many hours should I work?",
        "expected": "Correct assumption; 40 hours"
    },

    # 14. Context should override common knowledge
    {
        "name": "Strong Prior Conflict",
        "context": """
For the purposes of this policy, the standard workweek at
Nexora is thirty-two (32) hours.
""",
        "question": "How many hours is the standard workweek?",
        "expected": "32 hours"
    },

    # 15. Completely artificial context
    # Tests whether model REALLY follows context
    {
        "name": "Artificial Context Override",
        "context": """
For this policy, employees receive exactly seventeen (17)
days of Sick Leave per calendar year.
""",
        "question": "How many Sick Leave days do employees receive?",
        "expected": "17 days"
    },
]


# ============================================================
# RUN TESTS
# ============================================================

for i, test in enumerate(test_cases, 1):

    input_text = f"""Policy Context:
{test["context"].strip()}

Employee Question:
{test["question"]}"""

    answer = generate_answer(
        stage1_model,
        tokenizer,
        CTX_INSTR,
        input_text=input_text,
        max_new_tokens=100,
    )

    print("\n" + "=" * 100)
    print(f"TEST {i}: {test['name']}")
    print("=" * 100)

    print("\nCONTEXT:")
    print(test["context"].strip())

    print("\nQUESTION:")
    print(test["question"])

    print("\nEXPECTED:")
    print(test["expected"])

    print("\nMODEL:")
    print(answer)


TEST 1: Direct Fact

CONTEXT:
Full-time employees are entitled to twelve (12) days of
Sick Leave per calendar year.

QUESTION:
How many Sick Leave days do full-time employees get?

EXPECTED:
12 days

MODEL:
Full-time employees are entitled to twelve (12) days of Sick Leave per calendar year.

TEST 2: Context Conflict

CONTEXT:
Full-time employees are entitled to twelve (12) days of
Sick Leave per calendar year.

QUESTION:
I heard employees only get 10 Sick Leave days. Is that correct?

EXPECTED:
No; 12 days

MODEL:
The provided policy context does not specify this; it states twelve (12) sick leave days per calendar year for full-time employees.

TEST 3: Probation Fact

CONTEXT:
All new full-time hires are subject to a probationary period
of ninety (90) calendar days from their date of joining.

QUESTION:
How long is the probationary period?

EXPECTED:
90 calendar days

MODEL:
The probationary period for all new full-time hires is ninety (90) calendar days from their date of joining.



**--- Converting SFT-Based Model to GGUF ---**


In [ ]:
# Load the Stage 1 SFT merged model
stage1_gguf_model, stage1_gguf_tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE1_MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Convert to GGUF
# q4_k_m provides a good balance between model size, speed, and quality
stage1_gguf_model.save_pretrained_gguf(
    f"{STAGE1_MERGED_DIR}_gguf",
    stage1_gguf_tokenizer,
    quantization_method="q4_k_m",
)

print(f"Stage 1 SFT GGUF model saved to: {STAGE1_MERGED_DIR}_gguf")

# Free GPU memory
del stage1_gguf_model
del stage1_gguf_tokenizer
clear_gpu_memory()

In [ ]:
from huggingface_hub import login, HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError
hf_token = ""
repo_id = "pikabo88/Nexa_SFT"

if hf_token: # Only proceed if token is available
    # Login to Hugging Face
    login(token=hf_token, add_to_git_credential=True)
    api = HfApi()

try:
        api.upload_folder(
            folder_path=STAGE1_MERGED_DIR,
            repo_id=repo_id,
            repo_type="model",
            commit_message="Upload of final merged model",
            token=hf_token,
        )
        print(f"Successfully uploaded the merged model folder to {repo_id}")

        # Upload the GGUF model from its specific directory
        gguf_dir = "/content/NEXORA/stage1_instruction_merged_model_gguf"
        if os.path.exists(gguf_dir):
            api.upload_folder(
                folder_path=gguf_dir,
                repo_id=repo_id,
                repo_type="model",
                commit_message="Upload of GGUF model",
                token=hf_token,
            )
            print(f"Successfully uploaded the GGUF model folder to {repo_id}")
        else:
            print(f"GGUF directory not found: {gguf_dir}. Skipping GGUF upload.")

except Exception as e:
        print(f"An error occurred during model upload: {e}")
        print("Please verify your token permissions and repository existence.")

In [ ]:
from huggingface_hub import login, HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError
hf_token = ""
repo_id = "pikabo88/Nexa_SFT"

if hf_token: # Only proceed if token is available
    # Login to Hugging Face
    login(token=hf_token, add_to_git_credential=True)
    api = HfApi()

try:
        # Upload the GGUF model from its specific directory
        gguf_dir = "/content/NEXORA/stage1_instruction_merged_model_gguf" # Use the predefined variable
        if os.path.exists(gguf_dir):
            api.upload_folder(
                folder_path=gguf_dir,
                repo_id=repo_id,
                repo_type="model",
                commit_message="Upload of GGUF model",
                token=hf_token,
            )
            print(f"Successfully uploaded the GGUF model folder to {repo_id}")
        else:
            print(f"GGUF directory not found: {gguf_dir}. Skipping GGUF upload.")

except Exception as e:
        print(f"An error occurred during model upload: {e}")
        print("Please verify your token permissions and repository existence.")